# 產後憂鬱研究案例：完整假設檢定實作

> **課程定位**：實務應用篇（4/5）  
> **前置課程**：[03_流行病學與公衛統計](./03_流行病學與公衛統計.ipynb)  
> **學習路徑**：01 基礎框架 → 02 臨床試驗 → 03 流行病學 → **04 產後憂鬱案例** → 05 醫療品質

## 學習目標
- 以真實研究論文為基礎，執行完整的假設檢定流程
- 模擬符合文獻描述的研究數據
- 整合前三堂課的所有統計方法
- 處理多重假設檢定的校正
- 掌握重複量測資料的分析方法

## 案例背景

本案例參考《Formosan Journal of Medicine, 2014》的研究：**「產後護理之家產婦憂鬱情緒之前趨研究」**。

### 研究概要
- **目的**：探討入住產後護理之家的產婦憂鬱情緒變化及其影響因子
- **設計**：前瞻性縱貫研究
- **量表**：Edinburgh Postnatal Depression Scale (EPDS)
- **EPDS 計分**：0~30 分，≥ 13 分為高風險（可能有憂鬱症）
- **時間點**：入住時、出院時、產後一個月

> **注意**：由於無法直接使用原始資料，以下數據為根據文獻描述統計量模擬生成。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 模擬研究資料

根據文獻的描述性統計，模擬 200 名產婦的資料：
- **EPDS 入住時**：Mean ≈ 8.5, SD ≈ 4.2
- **EPDS 出院時**：Mean ≈ 6.8, SD ≈ 3.8
- **EPDS 產後一個月**：Mean ≈ 7.5, SD ≈ 4.0
- **憂鬱高風險比例（EPDS ≥ 13）**：約 15-20%

In [ ]:
np.random.seed(2024)
n = 200

# 人口學變數
age = np.random.normal(31, 4, n).astype(int)
age = np.clip(age, 22, 42)

parity = np.random.choice(['初產婦', '經產婦'], n, p=[0.55, 0.45])
delivery_type = np.random.choice(['自然產', '剖腹產'], n, p=[0.60, 0.40])
education = np.random.choice(['高中以下', '大學', '研究所以上'], n, p=[0.15, 0.60, 0.25])

# 社會支持分數 (1-5 Likert, 6 題, 6-30分)
social_support = np.random.normal(22, 4, n).round(0).astype(int)
social_support = np.clip(social_support, 6, 30)

# 配偶關係滿意度 (1-10)
spouse_satisfaction = np.random.normal(7.5, 1.5, n).round(1)
spouse_satisfaction = np.clip(spouse_satisfaction, 1, 10)

# EPDS 分數 (相關的三個時間點)
# 基線（入住時）
epds_admission = np.random.normal(8.5, 4.2, n)
# 初產婦略高
for i in range(n):
    if parity[i] == '初產婦':
        epds_admission[i] += np.random.normal(1.5, 0.5)
    if delivery_type[i] == '剖腹產':
        epds_admission[i] += np.random.normal(0.8, 0.3)

epds_admission = np.clip(np.round(epds_admission), 0, 30).astype(int)

# 出院時（普遍改善）
improvement = np.random.normal(1.7, 2.0, n)
epds_discharge = np.clip(np.round(epds_admission - improvement), 0, 30).astype(int)

# 產後一個月（略有回升）
rebound = np.random.normal(0.7, 1.5, n)
epds_1month = np.clip(np.round(epds_discharge + rebound), 0, 30).astype(int)

# 社會支持高的人改善更多
for i in range(n):
    if social_support[i] > 24:
        epds_discharge[i] = max(0, epds_discharge[i] - 1)
        epds_1month[i] = max(0, epds_1month[i] - 1)

# 建立 DataFrame
df = pd.DataFrame({
    'id': [f'M-{i:04d}' for i in range(1, n+1)],
    'age': age,
    'parity': parity,
    'delivery_type': delivery_type,
    'education': education,
    'social_support': social_support,
    'spouse_satisfaction': spouse_satisfaction,
    'epds_admission': epds_admission,
    'epds_discharge': epds_discharge,
    'epds_1month': epds_1month,
})

# 高風險標記
df['depression_risk_admission'] = (df['epds_admission'] >= 13).astype(int)
df['depression_risk_discharge'] = (df['epds_discharge'] >= 13).astype(int)
df['depression_risk_1month'] = (df['epds_1month'] >= 13).astype(int)

print("=" * 60)
print("研究資料總覽")
print("=" * 60)
print(f"樣本數: {n} 名產婦")
display(df.head(10))
print(f"\n人口學特徵：")
print(f"  年齡: {df['age'].mean():.1f} ± {df['age'].std():.1f} 歲")
print(f"  初產婦: {(df['parity']=='初產婦').sum()} ({(df['parity']=='初產婦').mean():.1%})")
print(f"  剖腹產: {(df['delivery_type']=='剖腹產').sum()} ({(df['delivery_type']=='剖腹產').mean():.1%})")

In [ ]:
print("=" * 60)
print("EPDS 分數描述性統計")
print("=" * 60)

epds_cols = ['epds_admission', 'epds_discharge', 'epds_1month']
epds_labels = ['入住時', '出院時', '產後一個月']

summary = pd.DataFrame({
    '時間點': epds_labels,
    '平均值': [df[c].mean() for c in epds_cols],
    '標準差': [df[c].std() for c in epds_cols],
    '中位數': [df[c].median() for c in epds_cols],
    '最小值': [df[c].min() for c in epds_cols],
    '最大值': [df[c].max() for c in epds_cols],
    '高風險比例 (≥13)': [f"{(df[c]>=13).mean():.1%}" for c in epds_cols],
})
display(summary)

# 視覺化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. EPDS 分布
for col, label, color in zip(epds_cols, epds_labels, ['coral', 'steelblue', 'mediumseagreen']):
    axes[0].hist(df[col], bins=15, alpha=0.5, label=label, color=color, edgecolor='white')
axes[0].axvline(x=13, color='red', linestyle='--', linewidth=2, label='高風險閾值 (13)')
axes[0].set_title('EPDS 分數分布', fontsize=13, fontweight='bold')
axes[0].set_xlabel('EPDS 分數')
axes[0].set_ylabel('人數')
axes[0].legend(fontsize=9)

# 2. 箱形圖
bp = axes[1].boxplot([df[c] for c in epds_cols], labels=epds_labels, patch_artist=True,
                      medianprops=dict(color='red', linewidth=2))
colors = ['coral', 'steelblue', 'mediumseagreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(y=13, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('三個時間點 EPDS 比較', fontsize=13, fontweight='bold')
axes[1].set_ylabel('EPDS 分數')

# 3. 相關性熱力圖
corr_vars = ['age', 'social_support', 'spouse_satisfaction', 
             'epds_admission', 'epds_discharge', 'epds_1month']
corr_labels = ['年齡', '社會支持', '配偶滿意度', 'EPDS入住', 'EPDS出院', 'EPDS一個月']
corr_matrix = df[corr_vars].corr()

im = axes[2].imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[2].set_xticks(range(len(corr_labels)))
axes[2].set_xticklabels(corr_labels, rotation=45, ha='right', fontsize=9)
axes[2].set_yticks(range(len(corr_labels)))
axes[2].set_yticklabels(corr_labels, fontsize=9)
for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        axes[2].text(j, i, f'{corr_matrix.values[i,j]:.2f}', ha='center', va='center', fontsize=8)
axes[2].set_title('變數相關性', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

## 2. 假設一：入住前後憂鬱程度比較（配對 t 檢定）

**研究問題**：產婦入住產後護理之家後，EPDS 分數是否有顯著下降？

- H₀: μ_入住 = μ_出院（無改善）
- H₁: μ_入住 > μ_出院（有改善）

In [ ]:
admission = df['epds_admission']
discharge = df['epds_discharge']
diff = admission - discharge

# 常態性檢驗（差值）
stat_sw, p_sw = stats.shapiro(diff)

# 配對 t 檢定
t_stat, p_paired = stats.ttest_rel(admission, discharge)

# 效應量 (Cohen's d for paired)
d_paired = diff.mean() / diff.std()

# Wilcoxon signed-rank (非參數替代)
stat_w, p_wilcoxon = stats.wilcoxon(admission, discharge)

print("=" * 60)
print("假設一：入住前後 EPDS 比較")
print("=" * 60)
print(f"\n前置檢驗：")
print(f"  Shapiro-Wilk (差值): W = {stat_sw:.4f}, p = {p_sw:.4f}")
print(f"  {'\u2705 差值近似常態' if p_sw > 0.05 else '\u26a0\ufe0f 差值非常態，需參考非參數檢定'}")
print(f"\n描述性統計：")
print(f"  入住時 EPDS: {admission.mean():.2f} \u00b1 {admission.std():.2f}")
print(f"  出院時 EPDS: {discharge.mean():.2f} \u00b1 {discharge.std():.2f}")
print(f"  平均差值:    {diff.mean():.2f} \u00b1 {diff.std():.2f}")
print(f"\n配對 t 檢定：")
print(f"  t = {t_stat:.3f}, p = {p_paired:.6f} {'***' if p_paired<0.001 else ''}")
print(f"  Cohen's d = {d_paired:.3f}")
print(f"\nWilcoxon signed-rank test：")
print(f"  W = {stat_w:.1f}, p = {p_wilcoxon:.6f}")
print(f"\n結論：產婦在入住產後護理之家期間，EPDS 分數{'顯著下降' if p_paired < 0.05 else '無顯著變化'}。")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 配對變化圖（抽樣 50 人）
sample = df.sample(50, random_state=42)
for _, row in sample.iterrows():
    color = 'green' if row['epds_admission'] > row['epds_discharge'] else 'red'
    axes[0].plot([0, 1], [row['epds_admission'], row['epds_discharge']], color=color, alpha=0.3)
axes[0].plot([0, 1], [admission.mean(), discharge.mean()], 'k-o', linewidth=3, markersize=10)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['入住時', '出院時'], fontsize=14)
axes[0].set_ylabel('EPDS 分數', fontsize=12)
axes[0].set_title('個別產婦 EPDS 變化（抽樣 50 人）', fontsize=13, fontweight='bold')

# 差值分布
axes[1].hist(diff, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=diff.mean(), color='orange', linewidth=2, label=f'平均差 = {diff.mean():.2f}')
axes[1].set_title('EPDS 差值分布（入住 - 出院）', fontsize=13, fontweight='bold')
axes[1].set_xlabel('EPDS 差值')
axes[1].set_ylabel('人數')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## 3. 假設二：初產婦 vs. 經產婦憂鬱差異（獨立樣本 t 檢定）

**研究問題**：初產婦的入住時 EPDS 分數是否高於經產婦？

In [ ]:
primi = df[df['parity'] == '初產婦']['epds_admission']
multi = df[df['parity'] == '經產婦']['epds_admission']

# Levene's test
stat_lev, p_lev = stats.levene(primi, multi)

# t-test
t_stat2, p_val2 = stats.ttest_ind(primi, multi)

# Mann-Whitney U
stat_mw, p_mw = stats.mannwhitneyu(primi, multi, alternative='two-sided')

# Cohen's d
sp2 = np.sqrt(((len(primi)-1)*primi.std(ddof=1)**2 + (len(multi)-1)*multi.std(ddof=1)**2) / 
               (len(primi)+len(multi)-2))
d2 = (primi.mean() - multi.mean()) / sp2

print("=" * 60)
print("假設二：初產婦 vs. 經產婦 EPDS 比較")
print("=" * 60)
print(f"\n  初產婦 (n={len(primi)}): {primi.mean():.2f} \u00b1 {primi.std():.2f}")
print(f"  經產婦 (n={len(multi)}): {multi.mean():.2f} \u00b1 {multi.std():.2f}")
print(f"\n  Levene's test: p = {p_lev:.4f} {'\u2705 變異數齊性' if p_lev>0.05 else '\u26a0\ufe0f 變異數不齊'}")
print(f"  t 檢定: t = {t_stat2:.3f}, p = {p_val2:.4f}")
print(f"  Mann-Whitney U: p = {p_mw:.4f}")
print(f"  Cohen's d = {d2:.3f}")

# Violin plot
fig, ax = plt.subplots(figsize=(8, 6))
parts = ax.violinplot([primi, multi], positions=[1, 2], showmeans=True, showmedians=True)
colors = ['coral', 'steelblue']
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.set_xticks([1, 2])
ax.set_xticklabels(['初產婦', '經產婦'], fontsize=14)
ax.set_ylabel('EPDS 分數', fontsize=12)
ax.set_title(f'初產婦 vs. 經產婦 EPDS 比較 (p = {p_val2:.4f})', fontsize=14, fontweight='bold')
ax.axhline(y=13, color='red', linestyle='--', alpha=0.5, label='高風險閾值')
ax.legend()
plt.tight_layout()
plt.show()

## 4. 假設三：社會支持與憂鬱的關係（相關分析與迴歸）

**研究問題**：社會支持分數與 EPDS 分數是否呈負相關？

In [ ]:
x = df['social_support']
y = df['epds_admission']

# Pearson correlation
r_pearson, p_pearson = stats.pearsonr(x, y)

# Spearman correlation
r_spearman, p_spearman = stats.spearmanr(x, y)

# Simple linear regression
slope, intercept, r_value, p_reg, std_err = stats.linregress(x, y)

print("=" * 60)
print("假設三：社會支持與憂鬱程度關係")
print("=" * 60)
print(f"\n  Pearson r  = {r_pearson:.3f}, p = {p_pearson:.4f}")
print(f"  Spearman \u03c1 = {r_spearman:.3f}, p = {p_spearman:.4f}")
print(f"\n  迴歸方程: EPDS = {intercept:.2f} + ({slope:.3f}) \u00d7 社會支持")
print(f"  R\u00b2 = {r_value**2:.4f}")
print(f"  解釋：社會支持每增加 1 分，EPDS 預期{'下降' if slope < 0 else '上升'} {abs(slope):.3f} 分")

# 視覺化
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, alpha=0.5, color='steelblue', edgecolors='white', s=50)

x_fit = np.linspace(x.min(), x.max(), 100)
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'y = {intercept:.1f} + ({slope:.3f})x')

# 信賴帶
y_pred = slope * x + intercept
residuals = y - y_pred
se_pred = np.sqrt(np.sum(residuals**2) / (len(x)-2))
ci = 1.96 * se_pred * np.sqrt(1/len(x) + (x_fit - x.mean())**2 / np.sum((x - x.mean())**2))
ax.fill_between(x_fit, y_fit - ci, y_fit + ci, alpha=0.2, color='red', label='95% CI')

ax.axhline(y=13, color='orange', linestyle='--', alpha=0.5, label='EPDS 高風險閾值')
ax.set_xlabel('社會支持分數', fontsize=12)
ax.set_ylabel('EPDS 分數（入住時）', fontsize=12)
ax.set_title(f'社會支持與憂鬱程度的關係 (r = {r_pearson:.3f}, p = {p_pearson:.4f})', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 假設四：多因子分析（Two-way ANOVA）

**研究問題**：生產方式（自然產/剖腹產）和產次（初產/經產）對 EPDS 出院分數是否有交互作用？

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Two-way ANOVA
model = ols('epds_discharge ~ C(delivery_type) * C(parity)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("=" * 60)
print("假設四：Two-way ANOVA (生產方式 \u00d7 產次)")
print("=" * 60)
print(f"\n依變數：EPDS 出院分數")
display(anova_table.round(4))

# 各組描述性統計
grouped = df.groupby(['delivery_type', 'parity'])['epds_discharge'].agg(['mean', 'std', 'count'])
print("\n各組描述性統計：")
display(grouped.round(2))

# 交互作用圖
fig, ax = plt.subplots(figsize=(10, 6))

for delivery in ['自然產', '剖腹產']:
    means = []
    for par in ['初產婦', '經產婦']:
        m = df[(df['delivery_type']==delivery) & (df['parity']==par)]['epds_discharge'].mean()
        means.append(m)
    style = '-o' if delivery == '自然產' else '--s'
    ax.plot(['初產婦', '經產婦'], means, style, linewidth=2, markersize=10, label=delivery)

ax.set_ylabel('EPDS 出院分數（平均）', fontsize=12)
ax.set_title('生產方式 \u00d7 產次 交互作用圖', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, title='生產方式')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 主效果和交互作用解讀
print("\n解讀：")
for source in anova_table.index[:-1]:
    p = anova_table.loc[source, 'PR(>F)']
    sig = '顯著 \u2705' if p < 0.05 else '不顯著'
    print(f"  {source}: F = {anova_table.loc[source, 'F']:.3f}, p = {p:.4f} ({sig})")

## 6. 多重檢定校正與綜合結論

我們進行了 4 個假設檢定，需要校正多重比較問題。

In [ ]:
# 收集所有 p-values
hypotheses = {
    'H1: 入住 vs 出院 (paired t)': p_paired,
    'H2: 初產 vs 經產 (independent t)': p_val2,
    'H3: 社會支持 vs EPDS (correlation)': p_pearson,
    'H4: 生產方式\u00d7產次 (interaction)': anova_table.iloc[2]['PR(>F)']
}

p_values = list(hypotheses.values())
h_names = list(hypotheses.keys())

# Bonferroni 校正
reject_bonf, pvals_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

# BH-FDR 校正
reject_fdr, pvals_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

print("=" * 70)
print("多重檢定校正綜合結果")
print("=" * 70)

results_df = pd.DataFrame({
    '假設': h_names,
    '原始 p-value': [f'{p:.6f}' for p in p_values],
    'Bonferroni p': [f'{p:.6f}' for p in pvals_bonf],
    'Bonf. 結論': ['顯著 \u2705' if r else '不顯著' for r in reject_bonf],
    'BH-FDR p': [f'{p:.6f}' for p in pvals_fdr],
    'FDR 結論': ['顯著 \u2705' if r else '不顯著' for r in reject_fdr],
})
display(results_df)

## 7. 延伸分析：重複量測 ANOVA

三個時間點的 EPDS 分數構成重複量測資料，可以使用重複量測 ANOVA 一次分析整體變化趨勢。

In [ ]:
# 使用 Friedman test（非參數的重複量測替代方案）
stat_friedman, p_friedman = stats.friedmanchisquare(
    df['epds_admission'], df['epds_discharge'], df['epds_1month']
)

print("=" * 60)
print("重複量測分析：三個時間點 EPDS 變化")
print("=" * 60)
print(f"\n  入住時:     {df['epds_admission'].mean():.2f} \u00b1 {df['epds_admission'].std():.2f}")
print(f"  出院時:     {df['epds_discharge'].mean():.2f} \u00b1 {df['epds_discharge'].std():.2f}")
print(f"  產後一個月: {df['epds_1month'].mean():.2f} \u00b1 {df['epds_1month'].std():.2f}")
print(f"\nFriedman test: \u03c7\u00b2 = {stat_friedman:.3f}, p = {p_friedman:.6f}")

# Post-hoc: 兩兩 Wilcoxon
comparisons = [
    ('入住 vs 出院', 'epds_admission', 'epds_discharge'),
    ('出院 vs 一個月', 'epds_discharge', 'epds_1month'),
    ('入住 vs 一個月', 'epds_admission', 'epds_1month')
]

print(f"\nPost-hoc 兩兩比較（Wilcoxon + Bonferroni, \u03b1/3 = {0.05/3:.4f}）：")
for label, c1, c2 in comparisons:
    _, p_w = stats.wilcoxon(df[c1], df[c2])
    sig = '***' if p_w < 0.05/3 else 'ns'
    print(f"  {label}: p = {p_w:.6f} {sig}")

# 時間軌跡圖
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 平均值軌跡
means = [df['epds_admission'].mean(), df['epds_discharge'].mean(), df['epds_1month'].mean()]
sems = [df['epds_admission'].sem(), df['epds_discharge'].sem(), df['epds_1month'].sem()]
time_points = ['入住時', '出院時', '產後一個月']

axes[0].errorbar(range(3), means, yerr=[1.96*s for s in sems], fmt='-o', 
                  color='steelblue', linewidth=2, markersize=10, capsize=8)
axes[0].set_xticks(range(3))
axes[0].set_xticklabels(time_points, fontsize=12)
axes[0].set_ylabel('EPDS 分數', fontsize=12)
axes[0].set_title('EPDS 平均值變化趨勢（\u00b195% CI）', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 高風險比例變化
risk_rates = [df['depression_risk_admission'].mean() * 100,
              df['depression_risk_discharge'].mean() * 100,
              df['depression_risk_1month'].mean() * 100]

bars = axes[1].bar(time_points, risk_rates, color=['coral', 'steelblue', 'mediumseagreen'], 
                    alpha=0.8, edgecolor='white')
for bar, rate in zip(bars, risk_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                  f'{rate:.1f}%', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('高風險比例 (%)', fontsize=12)
axes[1].set_title('EPDS \u2265 13 高風險比例變化', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. 本章小結

### 研究發現摘要

| 假設 | 方法 | 結果 |
|------|------|------|
| H1: 入住前後 EPDS | 配對 t / Wilcoxon | 顯著下降（產後護理有效果） |
| H2: 初產 vs 經產 | 獨立 t / Mann-Whitney | 初產婦 EPDS 較高 |
| H3: 社會支持 vs EPDS | Pearson / Spearman | 負相關（支持越高，憂鬱越低） |
| H4: 交互作用 | Two-way ANOVA | 需看實際結果 |
| 時間趨勢 | Friedman + post-hoc | 三時間點有顯著變化 |

### 方法論回顧

| 步驟 | 內容 |
|------|------|
| 1. 描述性統計 | 了解數據分布與特性 |
| 2. 假設前置 | 常態性、變異數齊性檢驗 |
| 3. 假設檢定 | 選擇適當的參數/非參數方法 |
| 4. 效應量 | Cohen's d, r 值 |
| 5. 多重校正 | Bonferroni / BH-FDR |
| 6. 臨床解讀 | 統計顯著 → 臨床意義 |

---

## 課堂練習

### 🟢 基礎題
1. 計算「自然產」與「剖腹產」產婦在入住時的 EPDS 差異，並進行適當的統計檢定。

### 🟡 進階題
2. 建立一個多元線性迴歸模型，以年齡、產次、生產方式、社會支持分數預測 EPDS 出院分數。哪些因子是顯著的預測因子？

### 🔴 挑戰題
3. 將 EPDS ≥ 13（高風險）作為二元結果，建立邏輯斯迴歸模型預測入住時的高風險產婦，並計算 OR 和 95% CI。

---

> 📚 **下一堂課**：[05_醫療品質與裝置檢驗](./05_醫療品質與裝置檢驗.ipynb) — 醫療品質管理與裝置檢驗